# 第5回空戦AIチャレンジ 初級チュートリアル

このノートブックは、配布データの説明である `README.md`(ルート) と、エージェント実装の説明である
`make_agent.ipynb`(ルート) の要点を1本にまとめ、**初めて触る人が最短で「自分のエージェントで対戦する」
ところまで到達できる**ようにしたものである。

| このノートブック | 詳しく知りたいとき |
| --- | --- |
| 配布データのどこを見ればよいか | ルートの `README.md` |
| Agent クラスに何を実装すればよいか | ルートの `make_agent.ipynb`(網羅的なリファレンス) |
| シミュレータの仕様(座標系・observables・commands) | `docs/html/core/index.html` |
| 強化学習で学習させたいとき | 本ノートブック §7 と `tutorial/RLAgent/` |

## 目次

0. [このチュートリアルの位置づけ](#0-このチュートリアルの位置づけ)
1. [配布データの地図](#1-配布データの地図)
2. [まず動かす](#2-まず動かす)
3. [Agent の処理の流れ](#3-agent-の処理の流れ)
4. [最小の Agent を書く](#4-最小の-agent-を書く)
5. [提出フォーマットにまとめる](#5-提出フォーマットにまとめる)
6. [自作 Agent で対戦する](#6-自作-agent-で対戦する)
7. [次のステップ](#7-次のステップ)

## 0. このチュートリアルの位置づけ

### 事前準備

配布データ一式を展開し、以後のコマンドはそのルート（`validate.py`、`dist/`、`tutorial/` がある
ディレクトリ）で実行する。推奨環境は **Ubuntu 22.04 と Python 3.10** である。Windows では
WSL2 上に Ubuntu を用意して実行する。Windows ネイティブ向けの Whl は配布されていない。

環境構築には Docker とローカル環境のどちらか一方を使う。評価環境に近い構成で確認したい場合は
Docker を推奨する。Apple Silicon Mac では `dist/macos/` または `dockerfiles/Dockerfile.macos` を使う。

### 環境構築

#### Docker を使う場合（推奨）

Docker Desktop または Docker Engine を導入する。配布時の `docker-compose.yml` は NVIDIA GPU 用である。
GPU を使わない場合は、`dockerfile` を `dockerfiles/Dockerfile.cpu` に変更し、`deploy:` 以下の GPU
割当設定も削除してから、次を実行する。

```bash
docker compose up -d
docker exec -it atla5 bash
cd /workspace
python -c "import ASRCAISim1; print('OK')"
```

Mac の Docker では `Dockerfile.cpu` ではなく `dockerfiles/Dockerfile.macos` を指定し、同様に
GPU 割当設定を削除する。Docker 上では MPS は利用できない。
Apple Silicon Mac では、MPS を利用できるローカル環境での構築を推奨する。

#### ローカル環境を使う場合

Python 3.10 の仮想環境を作成し、先に必要な Python ライブラリを導入する。`torch` は使用する CPU/CUDA/MPS
環境に対応する PyTorch の配布元の手順で `2.7.0` を導入する。続いて OS・CPU アーキテクチャに対応する
`dist/` の requirements をインストールする。Linux x86 では次のとおり。

```bash
python3.10 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install tensorflow==2.13.0 pandas==2.1.4 scikit-learn==1.6.1 lightgbm==4.5.0 timeout-decorator==0.5.0 opencv-python-headless==4.11.0.86
# torch==2.7.0 は使用する CPU/CUDA 環境に合わせて別途インストールする
python -m pip install torch==2.7.0
python -m pip install -r dist/linux/requirements.txt
python -m pip install protobuf==3.20.3
python -c "import ASRCAISim1; print('OK')"
```

Linux arm64 は `dist/linux_aarch64/requirements.txt`、Apple Silicon Mac は
`dist/macos/requirements.txt` を指定する。Apple Silicon Mac でローカル環境を構築する場合、CMake は `3.29.6` を使用する。
ソースからのビルドや macOS 固有の制約はルートの `README.md` を参照する。

> **学習速度の目安**: 強化学習の実装・設定・モデル構成に依存するが、シミュレータ自体は CPU で動作するため、
> GPU 性能より CPU 性能の影響が大きい場合がある。参考として、Ryzen 7 5700X / RTX 4070 SUPER の
> CUDA Docker 環境より、M4 MacBook Air のローカル MPS 環境の方が速く学習できた検証例がある。
> 実際の構成で短い学習を実行し、速度を比較して環境を選ぶとよい。

> **補足**: ローカル環境の構築は、リポジトリを開いたコーディングエージェントに依存関係の確認や
> インストール手順の実行を任せる方法も手軽である。

> ここで導入する Whl はシミュレータ本体と標準プラグイン用である。§7 の強化学習チュートリアルは
> `setup_workspace.py` が純 Python 部分をコピーして使うため、`r8_contest_sample` の Whl ビルドは不要である。

### 前提

上記の環境構築後、以下のコマンドが通れば準備完了である。

```bash
python -c "import ASRCAISim1; print('OK')"
```

### 到達点

このノートブックを最後まで進めると、次のことができるようになる。

1. 配布された `validate.py` で対戦を実行し、結果を読める
2. Agent クラスに実装すべきメソッドと、その役割が分かる
3. `tutorial/SimpleAgent/` を読んで、自分のエージェントに書き換えられる
4. 提出フォーマット(zip)にまとめられる

**強化学習そのものの解説は行わない。** ニューラルネットワークを使って学習させる方法は §7 で
入口だけ示すので、まずは「学習を使わないエージェントを1つ完成させて対戦する」ことを目標にする。

## 1. 配布データの地図

配布データは全体では大きいが、**最初に触るのは次の4つだけ**である。

| 場所 | 中身 | 最初に見るか |
| --- | --- | --- |
| `validate.py` | 対戦を実行するプログラム | ◎ |
| `Agents/BenchMark/` | 初期行動判断モデル。コンペのベンチマークとして参戦する対戦相手 | ◎ |
| `tutorial/SimpleAgent/` | **このチュートリアルで読む最小エージェント** | ◎ |
| `Agents/MyAgent/` | 自分のエージェントの既定の置き場所(配布時点では `BenchMark` と同じ中身) | ◎ |
| `replay.py` | 対戦ログ(`.dat`)から mp4 動画を作る | ○ |
| `docs/html/core/index.html` | 仕様ドキュメント(Doxygen)のトップページ | ○ |
| `make_agent.ipynb` | Agent 実装の詳細リファレンス | ○ |
| `common/` | 戦闘場面の設定ファイル。**編集しない** | △ |
| `results/` | 対戦結果ログの出力先(自動生成) | △ |
| `simulator/sample/modules/r8_contest_sample/` | 公式の強化学習 Agent 実装例 | 後で |
| `simulator/sample/scripts/R8Contest/HandyRLSample/` | 強化学習サンプル本体 | 後で |
| `dist/` | シミュレータ本体の whl。環境構築時のみ使う | 済 |

### ドキュメントの引き方

仕様で迷ったら `docs/html/core/index.html` を開く。特に次の3ページは繰り返し参照することになる。

| 知りたいこと | ページ |
| --- | --- |
| Agent が受け取れる observables と、出力すべき commands の一覧 | 第5回空戦AIチャレンジ向けチュートリアル / Agent が入出力 observables と commands の形式 |
| シミュレーション中に各メソッドが呼ばれる順序 | シミュレーションに関する説明 / シミュレーションの処理の流れ |
| 座標系(`MotionState` / `Track3D` / `Track2D`) | シミュレーションに関する説明 / MotionState, Track |

ドキュメント内リンクを有効にするには、リポジトリのルートを HTTP サーバで公開しておくとよい。

```bash
python -m http.server 5500
```

VSCode を使っている場合は拡張機能 [LiveServer](https://marketplace.visualstudio.com/items?itemName=ritwickdey.LiveServer)
を入れて右下の「Go Live」を押せば同じことができる。

## 2. まず動かす

理屈より先に、配布状態のまま対戦を1回走らせる。

`validate.py` はリポジトリのルートにあり、`--myagent-module-path` は
**実行時のカレントディレクトリからの相対パス**で指定する必要がある(内部でパス区切り文字を `.` に
置き換えてモジュールとして読み込むため、絶対パスを渡すと失敗する)。
このノートブックは `tutorial/` にあるので、まずルートへ移動しておく。

引数を何も指定しないと、`Agents/MyAgent`(既定では `BenchMark` と同じ中身)と `Agents/BenchMark` の
対戦、つまり**初期行動判断モデル同士の対戦**になる。まずはこれで環境が動くことを確認する。

`--visualize 1` を付けると実行中に動画が再生されるが、その分時間がかかる。
`--num-validation` は対戦回数(既定3回)なので、動作確認では1回に減らすとよい。

### Docker 環境で動画を表示する

Docker コンテナ内で可視化ウィンドウを表示するには、VNC と noVNC を使う。まず、コンテナ内の Terminal で次を実行する。

```bash
vncserver :1 -geometry 1500x900 -depth 24
websockify -D --web=/usr/share/novnc/ 8080 localhost:5901
```

ホスト側のブラウザで [http://localhost:8080/vnc.html](http://localhost:8080/vnc.html) にアクセスして VNC に接続する。
接続先の Terminal を開き、次を実行すると、VNC 画面内に対戦動画が表示される。

```bash
python validate.py --visualize 1
```

表示例は以下のとおり。

![VNC 経由での動画表示例](../figs/vnctest.gif)

In [ ]:
!python validate.py --num-validation 1 --visualize 1

### 結果の読み方

対戦結果は `results/{--myagent-id}/validation_results.json` に保存される(既定では `results/0/`)。
主な項目は次の通り。

| キー | 意味 |
| --- | --- |
| `own` | 自分の陣営の色("Red"=西軍 / "Blue"=東軍) |
| `win` / `loss` / `draw` | 勝ち / 負け / 引き分けの回数。合計が `--num-validation` の値になる |
| `status` / `message` | プログラムの実行ステータス。`"success"` / `"ok"` なら正常 |
| `details.{n}.endReason` | 終了理由(`ELIMINATION`=全滅、`TIMEUP`=時間切れ など) |
| `details.{n}.numAlives` | 生き残った戦闘機と護衛対象機の数 |
| `details.{n}.scores` | 各陣営の対戦スコア |
| `details.{n}.hit_rate` | 勝者の誘導弾の命中率(無駄弾にならなかった割合) |
| `details.{n}.time_bonus` | 決着までの速さに応じたボーナス(0.0〜0.1) |
| `details.{n}.error` | プログラム的なエラーを起こしたか。`true` なら無効試合 |
| `details.{n}.red_error_message` / `blue_error_message` | エラー内容。**自作エージェントのデバッグはまずここを見る** |

決着した対戦の勝者のスコアは `0.6 + 0.3 * hit_rate + time_bonus`、敗者には `1.0 - (勝者のスコア)`
が加算される。引き分けは双方 0.5、エラーによる無効試合は双方 0 となる。

次のセルで要点だけ抜き出して表示する。

In [1]:
import json
import os


def show_result(result_dir="./results", agent_id="0"):
    path = os.path.join(result_dir, agent_id, "validation_results.json")
    with open(path, encoding="utf-8") as f:
        r = json.load(f)

    print(f"自陣営 : {r['own']}")
    print(f"戦績   : {r['win']}勝 {r['loss']}敗 {r['draw']}分")
    print(f"状態   : {r['status']} ({r['message']})")
    print("-" * 62)
    for n, d in r["details"].items():
        print(f"[{n}] {d['endReason']:<12} scores={d['scores']} "
              f"alives={d['numAlives']} hit_rate={d['hit_rate']:.2f} "
              f"time_bonus={d['time_bonus']:.2f}")
        for key in ("red_error_message", "blue_error_message"):
            if d.get(key):
                print(f"     !! {key}: {d[key]}")


show_result()

FileNotFoundError: [Errno 2] No such file or directory: './results\\0\\validation_results.json'

### 動画が表示されない場合

`--visualize 1` で動画が出ない環境では、対戦動画情報(`.dat` ファイル)を
保存してから `replay.py` で mp4 に変換する。

```bash
python validate.py --num-validation 1 --visualize 0 --replay 1
xvfb-run python replay.py
```

`validate.py` で `--myagent-id` や `--result-dir` を指定した場合は、`replay.py` にも
それぞれ `--myagent-id` / `--movie-dir` として同じ値を渡すこと。

## 3. Agent の処理の流れ

ここからが本題である。**自分が作るのは Agent クラスと Policy の2つ**で、
シミュレータがそれらを次のように呼び出す。

```
                     ┌──────────── 1 step (既定 3.0 秒) ────────────┐
observables ────────>│ makeObs   observables を Observation に加工   │
(シミュレータが公開)   └──────────────────────┬───────────────────────┘
                                            │ Observation
                                            v
                              ┌─────────────────────────┐
                              │ policy   行動を決める    │  ← Agent の外側
                              └─────────────┬───────────┘
                                            │ action
                     ┌──────────────────────v───────────────────────┐
                     │ deploy    action を commands(機体への指令)に  │
                     └──────────────────────┬───────────────────────┘
                                            │ commands
                                            v
                                    機体(Fighter)が動く
```

大まかには `(validate) -> perceive -> makeObs -> deploy -> control -> behave` の順に呼ばれる。
正確な順序はドキュメントの「シミュレーションに関する説明 / シミュレーションの処理の流れ」を参照。

### 実装するメソッドと役割

| メソッド | 呼ばれる頻度 | 役割 | 必須 |
| --- | --- | --- | --- |
| `initialize` | 生成時に1回 | `modelConfig`(設定ファイル)の読み込み | 実質必須 |
| `validate` | 開始時に1回 | 他の登場物(Ruler 等)に依存する初期化 | 任意 |
| `perceive` | 1 tick ごと | observables を見て内部状態を更新 | 任意 |
| **`makeObs`** | 1 step ごと | observables → Observation | **必須** |
| **`observation_space`** | 1回 | `makeObs` が返す Observation の型 | **必須** |
| **`action_space`** | 1回 | policy が返す action の型 | **必須** |
| **`deploy`** | 1 step ごと | action を解釈して「どう動きたいか」を決める | **必須** |
| **`control`** | 1 tick ごと | commands(機体への指令)を出す | **実質必須**(下記) |
| `behave` | 1 tick ごと | commands のさらに細かい生成 | 任意 |
| `serializeInternalState` | 必要時 | 内部状態の保存(実験的機能) | 任意 |

1 tick はシミュレーションの最小時間単位で、本コンペティションでは **1.0 秒**。
1 step(行動判断の周期)は設定ファイルの `interval` で決まり、既定では **3.0 秒**である。

> ### ⚠ 最初にはまるところ: `commands` は `control` で出す
>
> 基底クラスの `Agent::control()` は、**毎 tick すべての parent の `commands` を空にリセットする。**
>
> ```cpp
> // simulator/src/Agent.cpp
> void Agent::control(){
>     BaseType::control();
>     for(auto &&e:parents){
>         commands[e.second->getFullName()]=nl::json::object();   // ← 毎 tick 消される
>     }
> }
> ```
>
> `deploy` は 1 step (既定3秒) に1回しか呼ばれない。そのため **`deploy` でしか `commands` を
> 書かないと、次の tick で消されてしまい**、機体側が指令を読めずに次のエラーで落ちる。
>
> ```
> asset.CONTROL() failed. asset=Red/RedF4/HumanIntervention
> [json.exception.type_error.304] cannot use at() with null
> ```
>
> したがって **`commands` は 1 tick ごとに呼ばれる `control` で出す。**
> `deploy` は「どう動きたいか」を決めてメンバ変数に持たせるだけにする。
> `simulator/sample/modules/r8_contest_sample/` の公式サンプルもこの構成になっている。
>
> (`def control(self): pass` とオーバーライドしてリセット自体を止め、`deploy` で
> `commands` を書く書き方もできるが、射撃フラグが立ちっぱなしになるなどの副作用に注意が必要。)

### 混同しやすい5つの用語

| 用語 | 意味 | 誰が作るか |
| --- | --- | --- |
| **observables** | シミュレータが公開する生の観測情報。階層構造の json | シミュレータ |
| **Observation** | observables を行動判断用に加工したもの。`makeObs` の戻り値 | **自分(`makeObs`)** |
| **action** | policy が返す行動。`deploy` の引数として渡ってくる | **自分(policy)** |
| **commands** | 機体への指令。目標進行方向・目標速度・射撃要否など | **自分(`deploy`)** |
| `decision` | Agent クラス間で共通の行動意図表現。模倣学習の教師役にする場合のみ必要 | 任意 |

observables は階層構造になっており、json pointer 形式で取り出す。

```python
parent.observables.at_p("/sensor/track")        # 味方全機のレーダ航跡(マージ済み)
parent.observables.at_p("/sensor/mws/track")    # 被弾警報(飛来する誘導弾の方向)
parent.observables.at_p("/weapon/missiles")     # 自機の誘導弾の状態
parent.observables["motion"]                    # 自機の運動状態
```

## 4. 最小の Agent を書く

`tutorial/SimpleAgent/agent.py` が、**7メソッドだけを実装した最小のエージェント**である。
やっていることは次の4行に尽きる。

1. 被弾警報が出ていたら、その方向に背を向けて逃げる
2. 彼機を探知していたら、その方向へ向かい、射程内なら撃つ
3. 何も見えていなければ、敵陣方向へ前進する
4. 場外ラインに近づいたら戦域中央側へ戻す(場外に留まると失格して負ける)

以下、`agent.py` を上から順に読んでいく。**全文は `tutorial/SimpleAgent/agent.py` を開いて確認すること**
(以下のコードは説明用の抜粋なので、そのまま実行はできない)。

### どちらのファイルを変更するか

ルールベース Agent では、**戦術の判断を変えるだけなら `__init__.py`、判断に使える情報や実行できる行動を変えるなら `agent.py`** を変更する。

| 変更したいこと | 変更先 | 例 |
| --- | --- | --- |
| 見えている情報を基に、どちらへ飛ぶか・いつ撃つかを変える | `__init__.py` の `SimpleRuleBasedPolicy.step()` | 射撃距離、警報時の回避、敵がいない時の探索、場外回避の優先度 |
| Policy のしきい値を外部設定で変えられるようにする | `__init__.py` と `args.json` | `fireRangeKm`、`marginKm` |
| 新しい観測情報を Policy に渡す | `agent.py` の `makeObs()` と `observation_space()` | 敵の速度・進行方向、複数の敵、残弾、ミサイルの状態 |
| 行動の選択肢や action の形式を増減する | `agent.py` の `action_space()` と `deploy()` | 高度指令、目標選択方式、旋回角・速度の選択肢 |
| 機体への commands の出し方や毎 tick の制御を変える | `agent.py` の `control()` | 高度維持、速度制御、射撃コマンド、座標変換 |
| Agent の共通設定を変える | 原則 `config.json`（読み込みは `agent.py` の `initialize()`） | `turnTable`、`speedTable`、高度上下限、同時射撃数 |

`__init__.py` の Policy は `makeObs()` が返した Observation と `action_space()` で宣言された action しか扱えない。したがって、**既存の Observation と action の範囲で戦術を変えるだけなら `agent.py` は基本的に変更不要**である。
一方、Observation/action のキー・形・意味を変えた場合は、`agent.py` と `__init__.py` の両方を対応させる。

### 4-1. クラスの定義 — S方式と M方式

エージェントには2つの形態がある。

| | 継承するクラス | 担当機の参照 | `isUserAgentSingleAsset` |
| --- | --- | --- | --- |
| **S方式**(分散) | `SingleAssetAgent` | `self.parent`(1機) | `True` |
| **M方式**(中央集権) | `Agent` | `self.parents`(dict) | `False` |

S方式は1機につき1つのエージェントが割り当てられる。扱うのが1機だけなのでコードが短くなり、
`action_space` も1機分をそのまま書けばよい。**初めて作るなら S方式が分かりやすい。**

M方式は編隊全体を1つのエージェントで操作するため、連携行動を直接学習させられるが、
`self.parents` のループが増え、`action_space` も機数分並べた `spaces.Tuple` にする必要がある。

なお `Agent` を継承しても `self.parents` は使えるので、M方式のクラスで1機だけ担当させることもできる。

```python
from ASRCAISim1.core import SingleAssetAgent

class SimpleAgent(SingleAssetAgent):
    ...
```

> **補足: S方式とM方式はクラスを共有できる**
>
> S方式かM方式かを決めているのは `isUserAgentSingleAsset` の戻り値であり、
> **Agent クラスがどちらを継承しているかではない。** 実際、公式の提出サンプル
> `simulator/sample/scripts/R8Contest/MinimumEvaluation/HandyRLSample01{S,M}` の差分は
> 次の3値だけで、Agent クラスと `model_config.yaml` は完全に共通である。
>
> ```
> __init__.py       : isUserAgentSingleAsset -> True (S) / False (M)
> agent_config.json : maxParentNum 1, maxFriendNum 4 (S)
>                     maxParentNum 4, maxFriendNum 1 (M)
> ```
>
> つまり `self.parents` のループで書いておけば、設定を変えるだけで両方式を試せる。
> `tutorial/RLAgent/` はこの形になっている。

### 4-2. `initialize` — 設定を読む

**何をする所か:** エージェント生成時に一度だけ呼ばれ、設定値を読み込み、以後の処理で使うメンバ変数の初期値を用意する。

ユーザによる初期化は**コンストラクタではなく `initialize` で行う**のが本シミュレータの作法である。
コンストラクタには `modelConfig`(モデルのパラメータセット)と `instanceConfig`(インスタンス固有の
パラメータセット)が渡され、同名のメンバ変数として保持されている。

`modelConfig` の実体は `config.json` の内容で、Factory に登録される際の以下の `config` の部分にあたる。

```json
{
    "Factory": {
        "Agent": {
            "modelName": {
                "class": "className",
                "config": { ...ここが modelConfig... }
            }
        }
    }
}
```

行動判断の周期は `modelConfig` の `interval` で指定する。`SimpleAgent/config.json` では次のように
1 step = 3.0 秒、`perceive`/`control`/`behave` は 1.0 秒間隔としている。

```json
"interval": { "unit": "time", "step": 3.0, "perceive": 1.0, "control": 1.0, "behave": 1.0 }
```

読み出しには `getValueFromJsonKRD` のようなユーティリティを使うと、キーが無い場合の既定値を
書けるので便利である(戻り値は Python のプリミティブ型になる)。

```python
    def initialize(self):
        super().initialize()          # 基底クラスの initialize は必ず呼ぶ
        self.own = self.getTeam()     # 自陣営 "Red" or "Blue"
        mc = self.modelConfig

        # 目標方位[deg]の選択肢。自陣営の進行方向(敵陣方向)を0度とし、右が正。
        self.turnTable = np.array(
            getValueFromJsonKRD(mc, "turnTable", self.randomGen,
                                [-180.0, -135.0, -90.0, -45.0, 0.0, 45.0, 90.0, 135.0]),
            dtype=np.float64)
        self.speedTable = np.array(
            getValueFromJsonKRD(mc, "speedTable", self.randomGen, [200.0, 270.0, 340.0]),
            dtype=np.float64)
        self.maxEnemyNum = int(getValueFromJsonKRD(mc, "maxEnemyNum", self.randomGen, 5))

        self.altMin = getValueFromJsonKRD(mc, "altMin", self.randomGen, 2000.0)
        self.altMax = getValueFromJsonKRD(mc, "altMax", self.randomGen, 15000.0)
        self.altitudeKeeper = AltitudeKeeper(mc().get("altitudeKeeper", {}))
```

`self.modelConfig` は `nljson` 型なので、`[]` でキーを指定すると `nljson` が返る。
Python のプリミティブ型として使いたい場合は上のように末尾に `()` を付ける。

### 4-3. `validate` — 戦域の寸法と陣営座標系

**何をする所か:** シミュレーション開始時に一度だけ呼ばれ、Ruler や担当機など、生成時には参照できない他の登場物から情報を取得して初期化する。

他の登場物に依存する初期化はここで行う。`SimpleAgent` では Ruler(ルール判定を行う登場物)から
戦域の寸法を取得し、**陣営座標系**への変換クラスを作っている。

陣営座標系とは、自陣営の進行方向が +x 方向になるよう z 軸まわりに回転させ、防衛ライン中央が
原点になるよう平行移動させた座標系である。これを使うと **Red でも Blue でも同じロジックで書ける**
ため、必ず用意しておくとよい。

```python
    def validate(self):
        rulerObs = self.manager.getRuler()().observables()   # 末尾の () で dict に変換
        self.dOut = rulerObs["dOut"]      # 戦域中心から場外ラインまでの距離[m]
        self.dLine = rulerObs["dLine"]    # 戦域中心から防衛ラインまでの距離[m]
        self.teamOrigin = TeamOrigin(self.own == rulerObs["eastSider"], self.dLine)
```

以降、`self.teamOrigin.relPtoB(v)` で慣性座標系→陣営座標系、`relBtoP(v)` でその逆変換ができる。

> **注意: `relPtoB` / `relBtoP` は回転のみで平行移動しない。**
> 平行移動込みの変換は `absPtoB` / `absBtoP` である
> (`simulator/user_plugins/BasicAgentUtility/src/util/TeamOrigin.cpp` を参照)。
> 公式サンプルは位置にも `relPtoB` を使っており、**戦域中心を原点とした陣営座標系**として
> 扱っている。`SimpleAgent` もそれに合わせている。
>
> 本コンペティションの戦域寸法は実測で **`dOut` = `dLine` = 150 km** である。
> 陣営座標系では敵陣方向(x)・横方向(y)ともに戦域中心から ±150 km が場外ラインになる
> (x は `dLine`、y は `dOut` が上限)。

### 4-4. `makeObs` — observables を Observation に加工する

**何をする所か:** シミュレータの生データ（observables）から、Policy が判断に使う Observation を組み立てる。ここで Observation に含めなかった情報は、`__init__.py` の Policy からは利用できない。

**ここがエージェントの性能を左右する最重要メソッドである。** observables から必要な情報を取り出し、
行動判断に使いやすい形に整える。

observables は json なので、そのままではベクトル演算ができない。次のようにオブジェクトへ
**復元**してから使うのが定石である。

| 復元するクラス | 元データ | 得られるもの |
| --- | --- | --- |
| `MotionState` | `observables["motion"]` | 位置・速度・姿勢、各種座標変換 |
| `Track3D` | `/sensor/track` の各要素 | 彼機の位置・速度。射撃目標の指定にも使う |
| `Track2D` | `/sensor/mws/track` の各要素 | 飛来する誘導弾の**方向のみ**(距離は分からない) |

`transformTo(self.getLocalCRS())` は行動判断用の座標系に揃えるための変換である。
本コンペティションの設定では NED(North-East-Down、地球の曲率を無視)となる。

`SimpleAgent` の Observation は3つのキーからなる。**すべて物理量そのまま**である。

| キー | 要素 | 内容 |
| --- | --- | --- |
| `own` | 8 | `[0]`敵陣方向の位置[km] `[1]`横方向の位置[km] `[2]`高度[km] `[3]`速さ[m/s] `[4]`進行方位[deg] `[5]`横方向の場外までの余裕[km] `[6]`前方の余裕[km] `[7]`後方の余裕[km] |
| `enemy` | 4 | `[0]`探知しているか(1/0) `[1]`最近傍の彼機の方位[deg] `[2]`水平距離[km] `[3]`高度[km] |
| `threat` | 2 | `[0]`被弾警報の有無(1/0) `[1]`誘導弾が居る方位[deg] |

`own[5]`〜`own[7]` の「場外までの余裕」は後から追加したものである。**これを入れないと
場外に留まって失格し、生き残っていても負ける**(実測で確認した。§6 参照)。
「勝つために必要な情報は Observation に入れる」という設計判断の実例になっている。

```python
    def makeObs(self):
        parent = self.parent
        own = np.zeros([8], dtype=np.float32)
        enemy = np.zeros([4], dtype=np.float32)
        threat = np.zeros([2], dtype=np.float32)
        self.lastTrackInfo = []

        if not parent.isAlive():
            # 撃墜されている場合 observables は更新されないので何も見えない
            return {"own": own, "enemy": enemy, "threat": threat}

        # --- 自機 ---
        myMotion = MotionState(parent.observables["motion"]).transformTo(self.getLocalCRS())
        pos = self.teamOrigin.relPtoB(myMotion.pos())   # 慣性座標系 -> 陣営座標系(回転のみ)
        vel = self.teamOrigin.relPtoB(myMotion.vel())
        own[0] = pos[0] / 1000.0     # 敵陣方向の位置[km] (戦域中心基準)
        own[1] = pos[1] / 1000.0     # 横方向の位置[km]
        own[2] = -pos[2] / 1000.0    # 高度[km] (陣営座標系は z 下向きなので符号反転)

        # 場外ラインまでの余裕[km]。場外に留まると罰則が累積し、
        # 一定量を超えると失格(endReason が PENALTY)になって負ける。
        own[5] = (self.dOut - abs(pos[1])) / 1000.0   # 横方向の余裕
        own[6] = (self.dLine - pos[0]) / 1000.0       # 前方(敵陣側)の余裕
        own[7] = (self.dLine + pos[0]) / 1000.0       # 後方(自陣側)の余裕
        ...

        # --- 彼機の航跡(味方全機ぶんがマージされて /sensor/track に入っている) ---
        self.lastTrackInfo = [Track3D(t).transformTo(self.getLocalCRS())
                              for t in parent.observables.at_p("/sensor/track")]
        sortTrack3DByDistance(self.lastTrackInfo, [myMotion], True)   # 近い順に並べる
        ...

        # --- 被弾警報 ---
        if parent.observables.contains_p("/sensor/mws/track"):
            for mObs in parent.observables.at_p("/sensor/mws/track"):
                mwsTrack = Track2D(mObs).transformTo(self.getLocalCRS())
                d = self.teamOrigin.relPtoB(mwsTrack.dir())
                threat[0] = 1.0
                threat[1] = degrees(atan2(d[1], d[0]))   # 誘導弾が居る方位[deg]
                break

        return {"own": own, "enemy": enemy, "threat": threat}
```

#### 注意点3つ

1. **戻り値は対戦ログに保存されるため、json 化できる形でなければならない。**
   `numpy.ndarray` は自動で `list` に変換されるのでそのまま返してよい。それ以外の型
   (`Track3D` など)はそのままでは返せないので、必要な値を取り出すか `nljson` を経由する。
2. **`observation_space` の定義と形を一致させること。** 不一致は学習時に問題になる。
3. **`SimpleAgent` では正規化していない。** このエージェントは学習しないので、Observation は
   物理量そのまま(km, m/s, degree)にしてある。見たままの値なので Policy が読みやすい。
   **ニューラルネットに入力する場合は正規化が必要**になる(→ `tutorial/RLAgent/` を参照)。

観測できる情報の一覧はドキュメントの「Agent が入出力 observables と commands の形式」にある。
主なものは次の通り。

1. 自分と味方の機体諸元(位置、速度、姿勢、角速度、残弾数)
2. 自分と味方の誘導弾諸元(位置、速度、目標ID、誘導状態)
3. 相手の機体諸元(**位置と速度のみ**)
4. 相手の誘導弾諸元(**方向のみ**)
5. 戦闘開始からの経過時間

### 4-5. `observation_space` と `action_space` — 型を宣言する

**何をする所か:** Observation と action のキー・形・値の範囲をシミュレータに宣言する。`makeObs` の戻り値、Policy の戻り値、`deploy` が読む action はこの宣言と一致させる。

`makeObs` が返す形と、policy が返す行動の形を宣言する。どちらも `gymnasium.spaces` を使う。

```python
    def observation_space(self):
        low, high = np.finfo(np.float32).min, np.finfo(np.float32).max
        return spaces.Dict({
            "own":    spaces.Box(low, high, shape=[8], dtype=np.float32),
            "enemy":  spaces.Box(low, high, shape=[4], dtype=np.float32),
            "threat": spaces.Box(low, high, shape=[2], dtype=np.float32),
        })

    def action_space(self):
        # S方式なので1機分をそのまま定義すればよい。
        # M方式ではこれを機数分並べた spaces.Tuple にする必要がある。
        return spaces.Dict({
            "turn":   spaces.Discrete(len(self.turnTable)),    # どの方位に向かうか
            "speed":  spaces.Discrete(len(self.speedTable)),   # どの速度で飛ぶか
            "target": spaces.Discrete(1 + self.maxEnemyNum),   # 0=撃たない, 1..=何番目の彼機を撃つか
        })
```

**行動の設計は自由だが、以下の2項目は必ずどこかで指定できるようにしておく必要がある。**

1. **機動** — 目標方位で指定する。基準方位を自機正面とするか自陣営の進行方向とするかは自分で決める
2. **射撃有無と射撃対象** — `0` を「射撃なし」、`1`〜`maxEnemyNum` を対応する航跡とする `Discrete` が定石

`action_space` は「制限時間内に行動を返せなかったときのフォールバック」にも使われる。
`--time-out`(既定 3.0 秒)を超えると、ここで定義した空間からランダムサンプリングされた行動が
使われるので、**どんな値が来ても `deploy` が落ちないように**しておくこと。

### 4-6. `deploy` — action を解釈する(1 step ごと)

**何をする所か:** Policy が選んだ抽象的な action（方位の番号、速度の番号、目標番号）を、目標方向・速度・射撃対象という具体的な制御意図へ変換する。1 step ごとに呼ばれる。

policy が返した `action` を受け取り、「どう動きたいか」を決めてメンバ変数に持たせる。
**ここでは `commands` を書かない**(§3 の警告を参照)。

```python
    def deploy(self, action):
        parent = self.parent
        if not parent.isAlive():
            return

        # (a) 目標進行方向: 陣営座標系での方位角 -> 慣性座標系のベクトル
        az = radians(float(self.turnTable[int(action["turn"])]))
        self.dstDir = self.teamOrigin.relBtoP(np.array([cos(az), sin(az), 0.0]))

        # (b) 目標速度
        self.dstV = float(self.speedTable[int(action["speed"])])

        # (c) 射撃の可否と目標
        self.launchFlag = False
        self.target = Track3D()
        tIdx = int(action["target"]) - 1      # 0 は「撃たない」なので1つずらす
        if 0 <= tIdx < len(self.lastTrackInfo):
            candidate = self.lastTrackInfo[tIdx]
            # isLaunchableAt: その航跡に対して今撃てるかを機体側が判定してくれる
            if parent.isLaunchableAt(candidate) and self._countFlyingMissiles() < self.maxSimulShot:
                self.launchFlag = True
                self.target = candidate
```

### 4-7. `control` — commands を出す(1 tick ごと)

**何をする所か:** `deploy` で決めた制御意図を、機体が読める `commands` にして毎 tick 出力する。高度制限など、現在の機体状態に応じて毎 tick 補正すべき処理もここに置く。

`deploy` が決めた内容を、機体への指令 `commands` として毎 tick 書き込む。

```python
    def control(self):
        parent = self.parent
        if not parent.isAlive():
            return
        myMotion = MotionState(parent.observables["motion"]).transformTo(self.getLocalCRS())

        # 高度が上下限に収まるようピッチ角を補正する(AltitudeKeeper を使用)。
        # 毎 tick 計算し直すので、現在高度に応じた滑らかな上昇・下降になる。
        dstDir = np.array(self.dstDir, dtype=np.float64)
        n = sqrt(dstDir[0] ** 2 + dstDir[1] ** 2)
        dstPitch = atan2(-dstDir[2], n)
        bottom = self.altitudeKeeper(myMotion, dstDir, self.altMin)
        minPitch = atan2(-bottom[2], sqrt(bottom[0] ** 2 + bottom[1] ** 2))
        top = self.altitudeKeeper(myMotion, dstDir, self.altMax)
        maxPitch = atan2(-top[2], sqrt(top[0] ** 2 + top[1] ** 2))
        dstPitch = max(minPitch, min(maxPitch, dstPitch))
        cs, sn = cos(dstPitch), sin(dstPitch)
        dstDir = np.array([dstDir[0] / n * cs, dstDir[1] / n * cs, -sn])

        # commands を書き込む
        originalMyMotion = MotionState(parent.observables["motion"])
        self.commands[parent.getFullName()] = {
            "motion": {
                # 計算は localCRS で行ったので、機体に渡す際は機体側の座標系に戻す
                "dstDir": originalMyMotion.dirAtoP(dstDir, myMotion.pos(), self.getLocalCRS()),
                "dstV": self.dstV,
            },
            "weapon": {
                "launch": self.launchFlag,
                "target": self.target.to_json(),
            },
        }

        # 射撃は1回だけ。立てたままにすると毎 tick 射撃要求を出し続けてしまう。
        self.launchFlag = False
```

#### つまずきやすい点

- **`commands` を `deploy` だけで書いてしまう。** 基底クラスの `control` が毎 tick 消すので
  機体が指令を読めずエラーになる(§3 の警告)。
- **座標系を戻し忘れる。** `makeObs`/`deploy`/`control` の計算は `localCRS` で行うのが楽だが、
  `commands` で機体に渡すときは `dirAtoP` で機体側の座標系に戻す必要がある。
- **射撃目標は `Track3D` オブジェクトで渡す。** `commands` では `to_json()` した形で入れる。
- **射撃フラグを下げ忘れる。** 立てたままだと毎 tick 射撃要求が出続ける。
- **高度制御を入れないと地面に激突する。** `AltitudeKeeper` を使って目標高度に向けたピッチ角の
  上下限を計算し、`dstDir` の z 成分を補正しておく。
- **同時射撃数を制限しないと弾切れする。** 飛翔中の誘導弾を数えて上限を設ける。
- **場外対策を入れないと、生き残っていても失格して負ける。**

## 5. 提出フォーマットにまとめる

提出物は「行動判断を実行するためのソースコード一式」を zip でまとめたものである。
必要なディレクトリ構造は次の通り。

```
SimpleAgent/
├── __init__.py     必須: 最初に呼ばれる init ファイル。4つの関数を定義する
├── args.json       必須: 各関数に渡す引数。ファイル名は必ず args.json
├── config.json     任意: Agent の設定(modelConfig)。名前は自由
└── agent.py        任意: Agent クラスの実装
```

学習済みモデル(`.pth` など)も、実行に必要なファイルはすべてこの中に含める。

### `__init__.py` に定義する4つの関数

```python
def getUserAgentClass(args={}):
    \"\"\"使用する Agent クラスを返す。\"\"\"
    from .agent import SimpleAgent
    return SimpleAgent


def getUserAgentModelConfig(args={}):
    \"\"\"Agent に渡す設定(modelConfig)を返す。\"\"\"
    with open(os.path.join(os.path.dirname(__file__), "config.json"), "r") as f:
        return json.load(f)


def isUserAgentSingleAsset(args={}):
    \"\"\"1機に1エージェントの S方式 なので True。M方式なら False。\"\"\"
    return True


def getUserPolicy(args={}):
    \"\"\"Observation から action を決める Policy を返す。\"\"\"
    return SimpleRuleBasedPolicy(args)
```

4つの関数はいずれも引数 `args` を取り、**`args.json` の内容が渡される**。
`SimpleAgent/args.json` では Policy のしきい値を外から調整できるようにしている。

```json
{ "fireRangeKm": 80.0, "marginKm": 25.0 }
```

配布されている `Agents/BenchMark/__init__.py` には `getBlueInitialState` というメソッドも
定義されているが、本コンペティションでは初期配置が常にランダムに決まるため呼び出されない
(過去回の名残であり、残しても削除しても動作に影響はない)。

### Policy — 行動を決める部分

`StandalonePolicy` を継承し、`step` メソッドを実装する。ここが「頭脳」にあたる部分で、
強化学習を使う場合はこの中でニューラルネットワークを動かすことになる。

```python
class SimpleRuleBasedPolicy(StandalonePolicy):
    def step(self, observation, reward, done, info,
             agentFullName, observation_space, action_space):
        own    = np.asarray(observation["own"],    dtype=np.float64)
        enemy  = np.asarray(observation["enemy"],  dtype=np.float64)
        threat = np.asarray(observation["threat"], dtype=np.float64)

        numSpeed = int(action_space["speed"].n)

        if threat[0] > 0.5:
            desiredAz = threat[1] + 180.0     # 1. 誘導弾に背を向けて逃げる
            speedIndex = numSpeed - 1
        elif enemy[0] > 0.5:
            desiredAz = enemy[1]              # 2. 探知した彼機の方位へ向かう
            speedIndex = numSpeed // 2
        else:
            desiredAz = 0.0                   # 3. 敵陣方向へ前進する
            speedIndex = numSpeed // 2

        # 4. 場外を避ける(最優先。上で決めた方位を上書きする)
        marginSide, marginFront, marginBack = own[5], own[6], own[7]
        if marginFront < self.marginKm:
            desiredAz = 90.0 if own[1] < 0.0 else -90.0   # 敵陣側に出過ぎ -> 横に流す
        elif marginBack < self.marginKm:
            desiredAz = 0.0                                # 下がり過ぎ -> 敵陣方向へ
        if marginSide < self.marginKm:
            desiredAz = -60.0 if own[1] > 0.0 else 60.0    # 横に出過ぎ -> 中央側へ

        targetIndex = 0
        if enemy[0] > 0.5 and enemy[2] <= self.fireRangeKm and threat[0] < 0.5:
            targetIndex = 1                   # 最も近い彼機を撃つ

        return {"turn": self._nearestTurnIndex(desiredAz, ...),
                "speed": speedIndex,
                "target": targetIndex}
```

`fireRangeKm` は `args.json` で 80 km にしている。実際に撃てるかどうかは Agent 側の
`parent.isLaunchableAt()` が判定するので、**しきい値は大きめにして機体の判定に任せる**のが
よい。小さすぎると近づく前に撃墜されて一発も撃てない(30 km にしていたときは実測で
射撃要求が0回だった)。

`makeObs` で Observation を物理量そのままにしておいたおかげで、Policy が
**「見たままの条件分岐」**で書けていることに注目してほしい。
Observation の設計とは、この条件分岐を書きやすくする作業でもある。

なお、行動空間の出力を確認したいだけなら `action_space.sample()` を返すダミー Policy を
作っておくとよい。

```python
class DummyPolicy(StandalonePolicy):
    def step(self, observation, reward, done, info,
             agentFullName, observation_space, action_space):
        action = action_space.sample()
        print(action)
        return action
```

## 6. 自作 Agent で対戦する

`--myagent-module-path` に自分のモジュールのパスを渡すと、初期行動判断モデル
(`Agents/BenchMark`)との対戦になる。

繰り返しになるが、**カレントディレクトリからの相対パスで指定すること**。

In [ ]:
!python validate.py --myagent-module-path tutorial/SimpleAgent --num-validation 3 --replay 1

In [ ]:
show_result()

### 期待される結果

`SimpleAgent` は 200 行あまりのルールベースなので、**初期行動判断モデルには勝てない。**
実測(オープン部門・Red 側・3回)ではこうなる。

```
own: Red | record: 0勝 3敗 0分 | status: success
[0] ELIMINATION  scores={'Blue': 0.771, 'Red': 0.229}  alives={'Blue': 5.0, 'Red': 0.0}
[1] ELIMINATION  scores={'Blue': 0.794, 'Red': 0.206}  alives={'Blue': 5.0, 'Red': 0.0}
[2] ELIMINATION  scores={'Blue': 0.752, 'Red': 0.248}  alives={'Blue': 5.0, 'Red': 1.0}
```

初期配置は毎回ランダムに決まるため結果は実行ごとに変わる。別の実行では
`TIMEUP` で 2 対 2 の引き分け(スコア 0.5 対 0.5)になり、彼機を1機撃墜できた対戦もあった。

**大事なのは `status` が `success` で `error` が `false` であること。**
負けていても、プログラムとして正しく動いていれば投稿できる。ここを出発点にして強くしていく。

`--color Blue` にしても同じように動く(陣営座標系を使っているため)。
`SimpleAgent` 同士で対戦させると `TIMEUP` で双方生き残る。

#### 場外対策を入れる前は何が起きたか

`own[5]`〜`own[7]`(場外までの余裕)を Observation に入れる前は、被弾警報のたびに 180 度反転して
逃げ続け、**戦域の後方から場外に出て失格していた**。

```
[0] PENALTY  alives={'Blue': 1.0, 'Red': 3.0}   ← 3機生き残っているのに敗北
```

`PENALTY` は場外滞留の罰則が累積して失格した状態である
(`simulator/user_plugins/BasicAirToAirCombatModels01/src/BasicAACRuler01.cpp`)。
戦域は 150 km 四方で、初期配置では**後方の余裕が 55 km 程度しかない**。
「勝つために必要な情報を Observation に入れる」ことの重要性がよく分かる例である。

### よく使うオプション

| オプション | 既定値 | 意味 |
| --- | --- | --- |
| `--myagent-module-path` | `Agents/MyAgent` | 自分のモジュールのパス(**相対パス**) |
| `--myagent-id` | `0` | 自分のエージェント名。結果の出力先 `results/{id}/` になる |
| `--opponent-module-path` | `Agents/BenchMark` | 対戦相手のモジュールのパス |
| `--num-validation` | `3` | 対戦回数 |
| `--color` | `Red` | 自分の陣営("Red"=西軍 / "Blue"=東軍)。**実際の対戦ではランダムに決まる** |
| `--youth` | `0` | ユース部門なら `1`、オープン部門なら `0`。**参加部門に合わせること** |
| `--replay` | `0` | `1` にすると対戦動画情報(`.dat`)を保存する |
| `--visualize` | `0` | `1` にすると実行中に動画を再生する(遅くなる) |
| `--time-out` | `3.0` | この秒数までに行動を返さないと行動空間からランダムサンプリングされる |
| `--memory-limit` | `7.5` | 使用メモリの上限[GB]。超えるとエラーで対戦が無効になる |
| `--make-log` | `1` | 各ステップの `observations` / `actions` / `infos` をログに残す |
| `--result-dir` | `./results` | 結果ログの出力先 |

`--color` を `Blue` にしても同じように戦えるか確認しておくこと。陣営座標系を使っていれば
同じロジックで動くはずである。

```bash
python validate.py --myagent-module-path tutorial/SimpleAgent --color Blue
```

### 応募用ファイルの作成

ディレクトリ構造を確認したうえで zip に固める。

In [ ]:
!cd tutorial && zip -r submit ./SimpleAgent -x "*__pycache__*"

### 投稿する前に確認すること

- **エラーなく実行できるか。** `validation_results.json` の `red_error_message` /
  `blue_error_message` が空であることを確認する
- **参加予定の部門の設定になっているか。** `--youth` の値を確認する
- **インターネット接続を行う処理を含めていないか。** 評価システム上では接続できない
- **実行時間とメモリが過大でないか。** 時間がかかりすぎるとランタイムエラーになる
- **`args.json` というファイル名になっているか。** 名前は固定である

## 7. 次のステップ

ここまでで「学習を使わないエージェントを作って対戦する」ことができた。次の進み方を挙げる。

### (a) Observation と行動の設計を作り込む

まずは `tutorial/SimpleAgent/` を書き換えて強くすることを勧める。
`makeObs` で見る情報を増やす(味方の位置、誘導弾の残数、経過時間など)、
`turnTable` を細かくする、といった改良が素直に効く。

観測できる情報の網羅的な一覧と、より丁寧な取得例は次を参照。

- ルートの `make_agent.ipynb` — Agent 実装の詳細リファレンス。`control` / `behave` /
  `perceive` / `serializeInternalState` や M方式の書き方も載っている
- `simulator/sample/modules/r8_contest_sample/r8_contest_sample/R8ContestPyAgentSample01.py`
  — 公式の実装例。observation の作り方が非常に細かく書かれている

### (b) 強化学習で学習させる

`tutorial/RLAgent/` に、**学習できる形にしたエージェント一式**を用意してある。
`SimpleAgent` との主な違いは次の3点である。

1. Observation を**正規化**している(ニューラルネットに入力するため)
2. `getUserPolicy` が `StandaloneHandyRLPolicy` を返す(重みファイル `.pth` を読み込む)
3. 学習に必要な設定ファイル(`model_config.yaml` / 報酬 / 学習設定)が付いている

> **本チュートリアルのフォルダ構成では、`r8_contest_sample` の Whl ビルドは不要である。**
> `setup_workspace.py` が、ニューラルネットワーク本体(`R8ContestTorchNNSampleForHandyRL`)など
> C++ に依存しない Python ファイルを学習用ディレクトリへコピーして利用する。
>
> ただし、公式パッケージ `r8_contest_sample` を直接 `import` して利用する場合は、
> C++ 拡張を含むため Whl ビルドが必要である。

> **環境に合わせた設定**
>
> 学習を始める前に `tutorial/seminar_config.yaml` を確認し、CPU コア数・メモリ量・利用する
> GPU に合わせて `worker.num_parallel`、`batch_size`、リプレイバッファの `capacity` などを調整する。
> Ray の `num_gpus` は CUDA GPU のリソース数であり、CUDA を使わない環境では `0` にする。
> `model_config` を変更した場合は、後述のとおり推論用設定も一致させること。
>
> **Ray 起動時の依存関係エラー**
>
> `google.protobuf.runtime_version` の ImportError により Ray の agent が起動しない場合、
> protobuf 3 系を使う環境では次の互換版を導入する。ほかの環境で問題なく起動する場合は不要である。
> ```bash
> python -m pip install "googleapis-common-protos==1.63.0" "google-api-core==2.15.0"
> ```

学習は HandyRL を用いた同梱サンプルで行う。次のセルで学習用の作業ディレクトリを
組み立てたうえで実行する(**このスクリプトは `tutorial/workspace/` を作るだけで、
配布物の他のファイルは変更しない**)。

```bash
python tutorial/setup_workspace.py
cd tutorial/workspace
python main.py seminar_config.yaml --train
```

学習済みモデルは `tutorial/workspace/results/Open/Single/<日時>/policies/checkpoints/Learner-latest.pth`
に保存される。これを `tutorial/RLAgent/` にコピーすれば(`args.json` の `weightPath` は
すでに `Learner-latest.pth` に設定済み)、§6 と同じ手順で対戦できる。

```bash
python validate.py --myagent-module-path tutorial/RLAgent --num-validation 3
```

> **注意: 学習時と推論時のネットワーク構成は一致させること**
>
> 推論時に使われる `RLAgent/model_config.yaml` の内容は、学習時の設定である
> `seminar_config.yaml` の `policy_config` → `Learner` → `model_config` と
> **完全に一致していなければならない。** 食い違うと構築されるネットワークの形が変わり、
> 学習済み重み(`.pth`)の読み込みに失敗する。
>
> そのため `RLAgent/model_config.yaml` は `seminar_config.yaml` から機械的に生成してあり、
> `setup_workspace.py` は実行のたびに両者の一致を検査して、食い違っていれば警告を出す。
> 学習設定の `model_config` を変更したら、警告に表示されるコマンドで作り直すこと。

学習設定の詳細はドキュメントの「HandyRL を用いた強化学習サンプル」を参照。
`seminar_config.yaml` の冒頭にも、まず触るとよい4箇所をコメントで書いてある。

### (c) 参考になる場所

| 目的 | 場所 |
| --- | --- |
| 公式の強化学習 Agent 実装例 | `simulator/sample/modules/r8_contest_sample/` |
| 強化学習サンプル本体(HandyRL) | `simulator/sample/scripts/R8Contest/HandyRLSample/` |
| 提出モジュールの実例 | `simulator/sample/scripts/R8Contest/MinimumEvaluation/HandyRLSample01S/` |
| 報酬の実装例 | `r8_contest_sample/R8ContestPyRewardSample01.py`, `...Sample02.py` |
| 初期行動判断モデルの説明 | `docs/html/core/page_r8_contest_initial_rulebased_agent.html` |
| 過去回のチュートリアル資料 | `docs/参考資料_第4回空戦AIチャレンジ/acac4_tutorial.pdf` |